## Transform Constructors Data
- Read bronze constructors table
- Keep only the columns required for analytics (drop url column)
- Standardise column names using snal case(constructorId -> constructor_id)
- Rename columns to make them more meaningful(name -> constructor_name)
- Remove duplicate records
- Transform values of columns nationality Title Case
- Write the transformed data to silver constructors table

In [0]:
%run ../00-common/01.environment_config

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.constructors'
silver_table = f'{catalog_name}.{silver_schema}.constructors'

###Step 1 - Read bronze circuits table

In [0]:
constructors_df = spark.table(bronze_table)
display(constructors_df)

### Step 2 - Keep only the columns required for analytics (drop url column)

In [0]:

from pyspark.sql import functions as F


In [0]:

constructors_dropped_df = constructors_df.drop('url')
display(constructors_dropped_df)

### Step 3
- Standardise column names using snal case(circuitId -> circuit_id)
- Rename columns to make them more meaningful(lat -> lattitude)

In [0]:
constructors_renamed_df = ( constructors_dropped_df
    .withColumnsRenamed(
        {
        'constructorId':'constructor_id',
        'name':'constructor_name'
        }
        )
)


### Step 6 - Remove duplicate records

In [0]:

#USING dropDuplicates(we can pass columns if needed) METHOD
constructors_distinct_df = constructors_renamed_df.dropDuplicates(["constructor_id"])

display(constructors_distinct_df)


### Step 7 -Transform values of columns circuit_name and locality to Title Case

In [0]:
constructors_final_df = (
    constructors_distinct_df
    .withColumn("nationality",F.initcap(F.col("nationality")))
    )
display(constructors_final_df)

### Step 8 - Write the transformed data to silver circuits table


In [0]:
(
    constructors_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)

)

In [0]:
display(spark.table(silver_table))